In [1]:
!pip install pytrec_eval

  Preparing metadata (setup.py) ... done
  Created wheel for pytrec_eval: filename=pytrec_eval-0.5-cp311-cp311-linux_x86_64.whl size=308656 sha256=558b1e7007b5e89d7664b07857c0e96c07eec708f6869c1b6f282e7077a5641e
  Stored in directory: /root/.cache/pip/wheels/0f/89/42/86aecdb99975f1840c27bc37fdfed72116abcf82e2c9dc76a8
Successfully built pytrec_eval


In [2]:
# Installing the ranx library
!pip install -q ranx



  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.0/859.0 kB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.0/135.0 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 4.4 MB/s eta 0:00:00


In [4]:
# Import ranx and fusion methods
from ranx import Run, Qrels, evaluate, fuse
import os
import urllib.request
import logging

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load the run files for the three models
run1 = Run.from_file(
    "/content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-cross-encoder-ms-marco-MiniLM-L-2-v2-2025-05-18_14-43-09/ranking.run",
    name="MiniLM",
    kind="trec"
)

run2 = Run.from_file(
    "/content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-cross-encoder-ms-marco-TinyBERT-L-2-v2-2025-05-18_16-10-19/ranking.run",
    name="TinyBERT",
    kind="trec"
)

run3 = Run.from_file(
    "/content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-distilroberta-base-2025-05-18_17-11-44/ranking.run",
    name="DistilRoBERTa",
    kind="trec"
)


In [ ]:
!wget https://msmarco.z22.web.core.windows.net/msmarcoranking/queries.tar.gz
!tar -xvzf queries.tar.gz

--2025-05-18 23:18:00--  https://msmarco.z22.web.core.windows.net/msmarcoranking/queries.tar.gz
Resolving msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)... 20.150.34.1
Connecting to msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)|20.150.34.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 18882551 (18M) [application/gzip]
Saving to: ‘queries.tar.gz’

queries.tar.gz      100%[===================>]  18.01M  34.1MB/s    in 0.5s    

2025-05-18 23:18:01 (34.1 MB/s) - ‘queries.tar.gz’ saved [18882551/18882551]

queries.dev.tsv
queries.eval.tsv
queries.train.tsv


In [6]:
# Download qrels file for TREC 2019 pass rate rankings
import requests

url = "https://trec.nist.gov/data/deep/2019qrels-pass.txt"
response = requests.get(url)

# Check the status code and save
if response.status_code == 200:
    with open("2019qrels-pass.txt", "w", encoding="utf-8") as f:
        f.write(response.text)
    print("Download complete: 2019qrels-pass.txt")
else:
    print(f"Download failed, status code：{response.status_code}")


Download complete: 2019qrels-pass.txt


In [ ]:
# Viewing documents in Colab
!ls -l 2019qrels-pass.txt

from google.colab import files
files.download("2019qrels-pass.txt")


-rw-r--r-- 1 root root 187092 May 18 23:35 2019qrels-pass.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from ranx import Qrels

# Loading a saved qrels file
qrels = Qrels.from_file("2019qrels-pass.txt", kind="trec")

print(f"qrels Loaded successfully！Total number of queries loaded：{len(qrels)}")



In [ ]:
from ranx import fuse, evaluate

runs = [run1, run2, run3]

fusion_methods = {
    "RRF": "rrf",
    "WMNZ": "wmnz",
    "LogISR": "log_isr",
    "BordaFuse": "bordafuse",
    "wCondorcet": "w_condorcet"
}

metrics = ["ndcg@10", "recall@100", "map@1000"]

for name, method in fusion_methods.items():
    print(f"\n🔷 Fusion Method: {name}")

    # Pass weights to the methods that need to be weighted
    if method in ["wmnz", "w_condorcet"]:
        fused = fuse(runs, method=method, params={"weights": [1.0, 1.0, 1.0]})
    else:
        fused = fuse(runs, method=method)

    scores = evaluate(qrels, fused, metrics)
    for m, score in scores.items():
        print(f"  {m}: {score:.4f}")





🔷 Fusion Method: RRF
  ndcg@10: 0.6911
  recall@100: 0.5063
  map@1000: 0.4502

🔷 Fusion Method: WMNZ
  ndcg@10: 0.6964
  recall@100: 0.5045
  map@1000: 0.4444

🔷 Fusion Method: LogISR
  ndcg@10: 0.6559
  recall@100: 0.4985
  map@1000: 0.4313

🔷 Fusion Method: BordaFuse
  ndcg@10: 0.6946
  recall@100: 0.5077
  map@1000: 0.4485

🔷 Fusion Method: wCondorcet
  ndcg@10: 0.6918
  recall@100: 0.5051
  map@1000: 0.4524


In [ ]:
from ranx import fuse, evaluate

# Three models of run
runs_all = {
    "MiniLM+TinyBERT": [run1, run2],
    "MiniLM+DistilRoBERTa": [run1, run3],
    "TinyBERT+DistilRoBERTa": [run2, run3]
}

metrics = ["ndcg@10", "recall@100", "map@1000"]

for name, run_pair in runs_all.items():
    print(f"\n🔷 Pair: {name}")

    # Use optimal method (WMNZ) with weight fusion
    fused = fuse(run_pair, method="wmnz", params={"weights": [1.0, 1.0]})

    scores = evaluate(qrels, fused, metrics)

    for m, score in scores.items():
        print(f"  {m}: {score:.4f}")


🔷 Pair: MiniLM+TinyBERT
  ndcg@10: 0.6943
  recall@100: 0.5038
  map@1000: 0.4511

🔷 Pair: MiniLM+DistilRoBERTa
  ndcg@10: 0.6743
  recall@100: 0.4940
  map@1000: 0.4245

🔷 Pair: TinyBERT+DistilRoBERTa
  ndcg@10: 0.6784
  recall@100: 0.4972
  map@1000: 0.4332


In [ ]:
import pandas as pd

# Read compressed files directly
query_path = "/content/drive/MyDrive/msmarco-test2019-queries.tsv.gz"
queries_df = pd.read_csv(query_path, sep='\t', header=None, names=["qid", "query"])
queries_df.head()




,qid,query
0,1108939,what slows down the flow of blood
1,1112389,"what is the county for grand rapids, mn"
2,792752,what is ruclip
3,1119729,what do you do when you have a nosebleed from ...
4,1105095,where is sugar lake lodge located


In [ ]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
import re

# Step 1: Load original queries (43 queries)
queries_df = pd.read_csv("/content/drive/MyDrive/msmarco-test2019-queries.tsv.gz", sep='\t', header=None, names=["qid", "query"])
queries_df["qid"] = queries_df["qid"].astype(str)

# Step 2: Define optimized CoT-style prompt

def build_prompt(query):
    return f"""Answer the following query:
{query}
Give the rationale before answering."""

# Step 3: Load LLM for expansion (Zephyr or similar)
generator = pipeline("text-generation", model="HuggingFaceH4/zephyr-7b-beta", max_new_tokens=64, device_map="auto")

# Step 4: Generate expansions using better prompting and filtering
expansions = []
for _, row in tqdm(queries_df.iterrows(), total=len(queries_df)):
    qid = row["qid"]
    query = row["query"]
    prompt = build_prompt(query)
    try:
        output = generator(prompt, do_sample=False)[0]["generated_text"]
        # Filter response: remove everything after "The final answer:" or similar
        filtered = re.split(r"(?i)The final answer:|Answer:", output)[0]
        filtered = filtered.replace(prompt, "").strip()
        # Combine original + expansion
        combined = f"{query} {filtered}".strip()
        expansions.append((qid, query, combined))
    except Exception as e:
        print(f"❌ Failed qid {qid}: {e}")
        expansions.append((qid, query, query))  # fallback to original

# Step 5: Save to TSV
expanded_df = pd.DataFrame(expansions, columns=["qid", "query", "expanded_query"])
expanded_df.to_csv("/content/drive/MyDrive/expanded_queries.tsv", sep="\t", index=False)

print("✅ Optimized expansion file saved to: expanded_queries.tsv")



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not in

...

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Device set to use cuda:0
100%|██████████| 200/200 [07:34<00:00,  2.27s/it]


✅ Optimized expansion file saved to: expanded_queries.tsv


In [ ]:
!wget https://trec.nist.gov/data/deep/2019qrels-pass.txt


--2025-05-21 21:14:32--  https://trec.nist.gov/data/deep/2019qrels-pass.txt
Resolving trec.nist.gov (trec.nist.gov)... 172.65.90.26, 172.65.90.24, 172.65.90.27, ...
Connecting to trec.nist.gov (trec.nist.gov)|172.65.90.26|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 187092 (183K) [text/plain]
Saving to: ‘2019qrels-pass.txt’

2019qrels-pass.txt  100%[===================>] 182.71K  --.-KB/s    in 0.1s    

2025-05-21 21:14:32 (1.76 MB/s) - ‘2019qrels-pass.txt’ saved [187092/187092]



In [ ]:
import pandas as pd
import gzip
from sentence_transformers.cross_encoder import CrossEncoder
from collections import defaultdict
import tqdm
import os

# Step 1: Load expanded_queries.tsv
expanded_df = pd.read_csv("/content/drive/MyDrive/expanded_queries.tsv", sep="\t")
queries = dict(zip(expanded_df["qid"].astype(str), expanded_df["expanded_query"]))

# Step 2: Load the qrels file (get the marked qid-pid relevance)）
relevant_docs = defaultdict(lambda: defaultdict(int))
with open("2019qrels-pass.txt") as f:
    for line in f:
        qid, _, pid, score = line.strip().split()
        if int(score) > 0:
            relevant_docs[qid][pid] = int(score)

# Step 3: Keep only query with annotations
relevant_qid = [qid for qid in queries if qid in relevant_docs]

# Step 4: Loading top1000 candidate documents
passage_cand = defaultdict(list)
with gzip.open("/content/drive/MyDrive/msmarco-passagetest2019-top1000.tsv.gz", 'rt', encoding='utf8') as f:
    for line in f:
        qid, pid, _, passage = line.strip().split("\t")
        if qid in relevant_qid:
            passage_cand[qid].append([pid, passage])

# Step 5: Using fine-tuning TinyBERT rerank
model_path = "/content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-cross-encoder-ms-marco-TinyBERT-L-2-v2-2025-05-18_16-10-19"
model = CrossEncoder(model_path, max_length=512)

run = {}
for qid in tqdm.tqdm(relevant_qid):
    query = queries[qid]
    cand = passage_cand[qid]
    pids = [x[0] for x in cand]
    corpus_sentences = [x[1] for x in cand]
    inputs = [[query, passage] for passage in corpus_sentences]
    scores = model.predict(inputs).tolist()
    run[qid] = {pid: score for pid, score in zip(pids, scores)}

# Step 6: Save as TREC run file
output_path = model_path + "/TinyBERT_expanded.run"
with open(output_path, "w") as f:
    for qid in run:
        sorted_scores = sorted(run[qid].items(), key=lambda x: x[1], reverse=True)
        for rank, (pid, score) in enumerate(sorted_scores):
            f.write(f"{qid} Q0 {pid} {rank} {score} TinyBERT_expanded\n")

print("✅ saved:", output_path)


100%|██████████| 43/43 [00:11<00:00,  3.72it/s]

✅ saved: /content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-cross-encoder-ms-marco-TinyBERT-L-2-v2-2025-05-18_16-10-19/TinyBERT_expanded.run


In [ ]:
import pytrec_eval
import pandas as pd

# Step 1: Loading qrels
with open("2019qrels-pass.txt") as f:
    qrels = pytrec_eval.parse_qrel(f)

# Step 2: Loading .run files (with exception line skipping)
def load_run(filepath):
    run = {}
    with open(filepath) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 6:
                continue
            try:
                qid = parts[0]
                pid = parts[2]
                score = float(parts[4])
                run.setdefault(qid, {})[pid] = score
            except ValueError:
                continue
    return run

run_expanded = load_run("/content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-cross-encoder-ms-marco-TinyBERT-L-2-v2-2025-05-18_16-10-19/TinyBERT_expanded.run")
run_original = load_run("/content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-cross-encoder-ms-marco-TinyBERT-L-2-v2-2025-05-18_16-10-19/ranking.run")

# Step 3: Evaluation with pytrec_eval
evaluator = pytrec_eval.RelevanceEvaluator(qrels, {'ndcg_cut_10', 'recall_100', 'map_cut_1000'})
res_orig = evaluator.evaluate(run_original)
res_exp = evaluator.evaluate(run_expanded)

# Step 4: Averaging assessment indicators
def mean_metrics(results):
    df = pd.DataFrame.from_dict(results, orient='index')
    return df.mean()

# Access to actual assessment indicator columns
df_orig = mean_metrics(res_orig)
df_exp = mean_metrics(res_exp)

# Constructing Comparison Results DataFrame
df_compare = pd.DataFrame({
    "Metric": df_orig.index,
    "Original": df_orig.values,
    "Expanded": df_exp.reindex(df_orig.index).values
})

print("🔍 TinyBERT original query vs Comparison of Extended Query Effectiveness：")
display(df_compare)




🔍 TinyBERT original query vs Comparison of Extended Query Effectiveness：


,Metric,Original,Expanded
0,recall_100,0.502914,0.497277
1,ndcg_cut_10,0.692802,0.662040
2,map_cut_1000,0.453425,0.441695


In [ ]:
import pandas as pd

# Read compressed files directly
query_path = "/content/drive/MyDrive/msmarco-test2019-queries.tsv.gz"
queries_df = pd.read_csv(query_path, sep='\t', header=None, names=["qid", "query"])
queries_df.head()


,qid,query
0,1108939,what slows down the flow of blood
1,1112389,"what is the county for grand rapids, mn"
2,792752,what is ruclip
3,1119729,what do you do when you have a nosebleed from ...
4,1105095,where is sugar lake lodge located


In [7]:
# 加载 top-1000 文档
import gzip
from collections import defaultdict

passage_cand = defaultdict(list)
with gzip.open("/content/drive/MyDrive/msmarco-passagetest2019-top1000.tsv.gz", 'rt', encoding='utf8') as f:
    for line in f:
        qid, pid, _, passage = line.strip().split("\t")
        passage_cand[qid].append((pid, passage))


In [8]:
import pandas as pd

queries_df = pd.read_csv("/content/drive/MyDrive/msmarco-test2019-queries.tsv.gz", sep="\t", header=None, names=["qid", "query"])
queries_df["qid"] = queries_df["qid"].astype(str).str.strip()


In [9]:
from sentence_transformers.cross_encoder import CrossEncoder
from tqdm import tqdm
import pandas as pd

# Step 0: Load 43 query ids in qrels
qrels_qids = set()
with open("2019qrels-pass.txt") as f:
    for line in f:
        qid, _, _, score = line.strip().split()
        if int(score) > 0:
            qrels_qids.add(qid)

# Step 1: Load queries (filter 43)
queries_df = pd.read_csv("/content/drive/MyDrive/msmarco-test2019-queries.tsv.gz", sep="\t", header=None, names=["qid", "query"])
queries_df["qid"] = queries_df["qid"].astype(str).str.strip()
queries_df = queries_df[queries_df["qid"].isin(qrels_qids)]

# Step 2: Initialising the model
model_path = "/content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-cross-encoder-ms-marco-TinyBERT-L-2-v2-2025-05-18_16-10-19"
model = CrossEncoder(model_path, max_length=512)

# Step 3: Extract Top-3 paragraph
top3_passages_per_q = {}
for _, row in tqdm(queries_df.iterrows(), total=len(queries_df)):
    qid = row["qid"]
    query = row["query"]
    if qid not in passage_cand:
        continue
    passages = passage_cand[qid]
    inputs = [[query, p[1]] for p in passages]
    scores = model.predict(inputs).tolist()
    pid_score = sorted(zip(passages, scores), key=lambda x: x[1], reverse=True)
    top3_passages = [x[0][1] for x in pid_score[:3]]
    top3_passages_per_q[qid] = (query, top3_passages)

print("✅ Top-3 Document extraction done. Total queries:", len(top3_passages_per_q))






100%|██████████| 43/43 [00:11<00:00,  3.80it/s]

✅ Top-3 Document extraction done. Total queries: 43


In [10]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
import re


# Step 2: Define CoT/PRF-style prompt
def build_prf_prompt(query, passages):
    context = "\n".join(passages)
    return f"""Given the following documents:

{context}

Refine the following query using the documents above:
{query}

Give the rationale before answering."""


# Step 3: Load LLM for expansion (Zephyr or similar)
generator = pipeline("text-generation", model="HuggingFaceH4/zephyr-7b-beta", max_new_tokens=90, device_map="auto")

# Step 4: Generate expansions with PRF
expansions = []

for qid, (query, passages) in tqdm(top3_passages_per_q.items()):
    prompt = build_prf_prompt(query, passages)
    try:
        output = generator(prompt, do_sample=False)[0]["generated_text"]
        # Filter response: remove everything after "The final answer:" or similar
        filtered = re.split(r"(?:The final answer:|Answer:)", output)[0]
        filtered = filtered.replace(prompt, "").strip()
        combined = f"{query} {filtered}".strip()
        expansions.append((qid, query, combined))
    except Exception as e:
        print(f"❌ Failed qid {qid}: {e}")
        expansions.append((qid, query, query))  # fallback to original

# Step 5: Save to TSV
expanded_df = pd.DataFrame(expansions, columns=["qid", "query", "expanded_query"])
expanded_df.to_csv("/content/drive/MyDrive/expanded_queries_prf.tsv", sep="\t", index=False)

print("✅ PRF-based expansion file saved to: expanded_queries_prf.tsv")



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not in

...

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Device set to use cuda:0
100%|██████████| 43/43 [00:20<00:00,  2.10it/s]


The new version of the PRF extended query file has been saved as: expanded_queries_prf1.tsv


In [14]:
import pandas as pd
import gzip
from sentence_transformers.cross_encoder import CrossEncoder
from collections import defaultdict
import tqdm
import os

# Step 1: Load expanded_queries_prf.tsv
expanded_df = pd.read_csv("/content/drive/MyDrive/expanded_queries_prf.tsv", sep="\t")
queries = dict(zip(expanded_df["qid"].astype(str), expanded_df["expanded_query"]))

# Step 2: Load the qrels file (get the marked qid-pid relevance)
relevant_docs = defaultdict(lambda: defaultdict(int))
with open("2019qrels-pass.txt") as f:
    for line in f:
        qid, _, pid, score = line.strip().split()
        if int(score) > 0:
            relevant_docs[qid][pid] = int(score)

# Step 3: Keep only queries with annotations
relevant_qid = [qid for qid in queries if qid in relevant_docs]

# Step 4: Load top1000 candidate documents
passage_cand = defaultdict(list)
with gzip.open("/content/drive/MyDrive/msmarco-passagetest2019-top1000.tsv.gz", 'rt', encoding='utf8') as f:
    for line in f:
        qid, pid, _, passage = line.strip().split("\t")
        if qid in relevant_qid:
            passage_cand[qid].append([pid, passage])

# Step 5: Using fine-tuned TinyBERT for rerank
model_path = "/content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-cross-encoder-ms-marco-TinyBERT-L-2-v2-2025-05-18_16-10-19"
model = CrossEncoder(model_path, max_length=512)

run = {}
for qid in tqdm.tqdm(relevant_qid):
    query = queries[qid]
    cand = passage_cand[qid]
    pids = [x[0] for x in cand]
    corpus_sentences = [x[1] for x in cand]
    inputs = [[query, passage] for passage in corpus_sentences]
    scores = model.predict(inputs).tolist()
    run[qid] = {pid: score for pid, score in zip(pids, scores)}

# Step 6: Save as TREC run file
output_path = model_path + "/TinyBERT_expanded_prf.run"
with open(output_path, "w") as f:
    for qid in run:
        sorted_scores = sorted(run[qid].items(), key=lambda x: x[1], reverse=True)
        for rank, (pid, score) in enumerate(sorted_scores):
            f.write(f"{qid} Q0 {pid} {rank+1} {score} TinyBERT_expanded_prf\n")

print("✅ PRF reranked run saved:", output_path)




100%|██████████| 43/43 [00:12<00:00,  3.39it/s]

✅ PRF reranked run saved: /content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-cross-encoder-ms-marco-TinyBERT-L-2-v2-2025-05-18_16-10-19/TinyBERT_expanded_prf.run


In [15]:
import pytrec_eval
import pandas as pd

# Step 1: Load qrels
with open("2019qrels-pass.txt") as f:
    qrels = pytrec_eval.parse_qrel(f)

# Step 2: Load .run files
def load_run(filepath):
    run = {}
    with open(filepath) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 6:
                continue
            try:
                qid = parts[0]
                pid = parts[2]
                score = float(parts[4])
                run.setdefault(qid, {})[pid] = score
            except ValueError:
                continue
    return run

run_original = load_run("/content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-cross-encoder-ms-marco-TinyBERT-L-2-v2-2025-05-18_16-10-19/ranking.run")
run_expanded = load_run("/content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-cross-encoder-ms-marco-TinyBERT-L-2-v2-2025-05-18_16-10-19/TinyBERT_expanded.run")
run_prf = load_run("/content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-cross-encoder-ms-marco-TinyBERT-L-2-v2-2025-05-18_16-10-19/TinyBERT_expanded_prf.run")

# Step 3: Evaluate
evaluator = pytrec_eval.RelevanceEvaluator(qrels, {'ndcg_cut_10', 'recall_100', 'map_cut_1000'})
res_orig = evaluator.evaluate(run_original)
res_exp = evaluator.evaluate(run_expanded)
res_prf = evaluator.evaluate(run_prf)

# Step 4: Average
def mean_metrics(results):
    df = pd.DataFrame.from_dict(results, orient='index')
    return df.mean()

df_orig = mean_metrics(res_orig)
df_exp = mean_metrics(res_exp)
df_prf = mean_metrics(res_prf)

# Step 5: Comparison
df_compare = pd.DataFrame({
    "Metric": df_orig.index,
    "Original": df_orig.values,
    "Expanded": df_exp.reindex(df_orig.index).values,
    "PRF_Expanded": df_prf.reindex(df_orig.index).values
})

print("📊 TinyBERT Original vs Expanded vs PRF Expanded:")
display(df_compare)


📊 TinyBERT Original vs Expanded vs PRF Expanded:


,Metric,Original,Expanded,PRF_Expanded
0,recall_100,0.502914,0.497277,0.456998
1,ndcg_cut_10,0.692802,0.662040,0.652669
2,map_cut_1000,0.453425,0.441695,0.408072
